<a href="https://colab.research.google.com/github/anjicx/CNHypergraph/blob/main/FirstScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Loading the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
#loading datasets
path = "/content/drive/MyDrive/PatientData/table_diagnoses.csv"

diagnosis = pd.read_csv(path, sep=";")
print(diagnosis.head())
stays = pd.read_csv("/content/drive/MyDrive/PatientData/final_one_percent_stays.csv", sep=";")
stays_secondaries = pd.read_csv("/content/drive/MyDrive/PatientData/final_one_percent_stays_secondaries.csv", sep=";")
age = pd.read_csv( "/content/drive/MyDrive/PatientData/table_age.csv",sep=";",encoding="latin1")

Mounted at /content/drive
   diagnose_id                          descr icd_code
0        14648               Pfannenlockerung     1010
1        14649            OsteolyseAcetabulum     1011
2        14650  GroßerKnochendefektAcetabulum     1012
3        14651   ChondropathiebeiKopfprothese     1013
4        14652       FehlpositionierungPfanne     1014


1. PATIENT FILTERING
Making a cohort of patients who have at least 3 unique diagnosis and their first appearance is from 20 age

In [ ]:
#Patient stays filtered
stays = stays.merge(age, on="ag_id", how="left")
stays = stays.rename(columns={"age": "age_group"})

patient_stays = stays[["patient_no", "entry_date", "exit_date", "sex_id", "age_group"]].copy()
patient_stays = patient_stays[patient_stays["sex_id"].isin([1, 2])].copy()

patient_stays["entry_date"] = pd.to_datetime(patient_stays["entry_date"])
patient_stays["exit_date"] = pd.to_datetime(patient_stays["exit_date"])
patient_stays = patient_stays.sort_values(["patient_no", "entry_date"])#patient stays sorted by entry date


#just takes the year from the entry date
patient_stays["year"] = patient_stays["entry_date"].dt.year
#patient_stays
#COUNTNING NUMBER OF UNIQUE YEARS IN VISITS OF EACH PATIENT
unique_years = (
    patient_stays.groupby("patient_no")["year"].nunique()# group by patient, count distinct years
    .reset_index(name="n_unique_years")
)

  # assign age group and sex from the first visit
first_visit = patient_stays.drop_duplicates("patient_no", keep="first")[
    ["patient_no", "sex_id", "age_group"]
]

patient_filter = first_visit.merge(unique_years, on="patient_no", how="left")

patient_filter["sex"] = patient_filter["sex_id"].map({
    1: "Male",
    2: "Female"
})

# FIRST CONDITION:20+ AGE GROUP
# age group 20+. Counting first age group in the dataset-to be 20+? 40 bis 44 Jahre lake this is age_group
patient_filter["age_group"] = patient_filter["age_group"].astype(str).str.strip()
patient_filter["age_start"] = (
    patient_filter["age_group"]
    .str.extract(r"(\d+)")
    .astype(float)
)# to extract the start age in the age group
patient_filter = patient_filter[patient_filter["age_start"] >= 20].copy()#filter 20+

# SECOND CONDITION NUMBER OF UNIQUE YEARS FOR THAT PATIENT MORE THEN 3
patient_filter = patient_filter[patient_filter["n_unique_years"] >= 3].copy()
#ALL THE PATIENTS THAT CAME THROUGH FILTER

In [ ]:
unique_years.head(5)

,patient_no,n_unique_years
0,1,5
1,3,3
2,4,2
3,5,1
4,7,2


In [ ]:
first_visit.head(5)

,patient_no,sex_id,age_group
0,1,2,10 bis 14 Jahre
2,3,1,05 bis 09 Jahre
3,4,1,50 bis 54 Jahre
4,5,1,15 bis 19 Jahre
6,7,1,10 bis 14 Jahre


patient_filter is our cohort of patients -> filtering only the stays from our cohort

In [ ]:
filtered_stays = stays[stays["patient_no"].isin(patient_filter["patient_no"])].copy()
filtered_stays["entry_date"] = pd.to_datetime(filtered_stays["entry_date"])
filtered_stays["exit_date"] = pd.to_datetime(filtered_stays["exit_date"], errors="coerce")

print("Filtered stays shape:", filtered_stays.shape)

Filtered stays shape: (264533, 13)


2. Diagnosis nodes

Creating nodes of diagnosis(dictionary containing all diagnosis), and table of stays and connected diagnosis to that stay.Each row is one diagnosis connected to that stay

In [ ]:
# building nodes of diagnosis

nodes = diagnosis.copy()
nodes["diagnose_id"] = pd.to_numeric(nodes["diagnose_id"])
nodes["node_id"] = range(len(nodes))
nodes = nodes[["node_id", "diagnose_id", "descr", "icd_code"]].copy()



#creating a tbl of primary diagnosis

primary = filtered_stays[["stay_id", "patient_no", "entry_date", "pri_diag_id"]].copy()#stays has for key to diagnosis
primary = primary.rename(columns={"pri_diag_id": "diagnose_id"})#rename foreign key diagnose_id
primary["diagnose_id"] = pd.to_numeric(primary["diagnose_id"])
primary["role"] = "primary"



#creating a tbl of secondary diagnosis

filtered_stays_secondaries = stays_secondaries[stays_secondaries["stay_id"].isin(filtered_stays["stay_id"])].copy()
filtered_stays_secondaries["sec_diag_id"] = pd.to_numeric(filtered_stays_secondaries["sec_diag_id"])
secondary = filtered_stays_secondaries.merge(filtered_stays[["stay_id", "patient_no", "entry_date"]],on="stay_id",how="inner")
secondary = secondary.rename(columns={"sec_diag_id": "diagnose_id"})
secondary["role"] = "secondary"



# NODES TABLE: node id connected to diagnosis
stay_diagnoses = pd.concat([primary, secondary], ignore_index=True)
# remove rows with missing diagnosis ids
stay_diagnoses = stay_diagnoses.dropna(subset=["diagnose_id"]).copy()
# inner join
stay_diagnoses = pd.merge(stay_diagnoses,nodes,on="diagnose_id",how="inner")

stay_diagnoses.head(5)


,stay_id,patient_no,entry_date,diagnose_id,role,node_id,descr,icd_code
0,4005,16,2015-07-08,1281,primary,1079,BösartigeNeubildung-Sonstigeundnichtnäherbezei...,C725
1,4100,17,2015-11-12,626,primary,516,"Kandidose,nichtnäherbezeichnet,Sooro.n.A.",B379
2,4665,19,2015-04-10,346,primary,319,SonstigenäherbezeichneteSpirochäteninfektionen,A698
3,4708,20,2015-03-26,1458,primary,1250,MelanomainsituansonstigenLokalisationen,D038
4,4904,21,2015-09-17,203,primary,215,Meningokokkenmeningitis(G01*),A390


For each patient keep only the first appearance of diagnosis.Then merge the whole data and sort so you can track which diagnosis was first.Afterwards divide by gender.

In [ ]:
patient_first_diag = (stay_diagnoses.groupby(["patient_no", "node_id"], as_index=False)["entry_date"].min()
.rename(columns={"entry_date": "first_date"})
)
patient_first_diag = patient_first_diag.merge(
    nodes[["node_id", "diagnose_id", "descr", "icd_code"]].drop_duplicates(subset=["node_id"]),
    on="node_id",
    how="left")
# sort for each patient diagnosis order
patient_first_diag = patient_first_diag.sort_values(
    ["patient_no", "first_date", "node_id"]
).copy()


In [ ]:
patient_sex = patient_filter[["patient_no", "sex_id", "sex"]].drop_duplicates().copy()
patient_first_diag_sex = patient_first_diag.merge(patient_sex,on="patient_no",how="left")

# sorted for diagnosis timestamp order, unique diagnosis per gender
male_diag = patient_first_diag_sex[patient_first_diag_sex["sex"] == "Male"].copy()
female_diag = patient_first_diag_sex[patient_first_diag_sex["sex"] == "Female"].copy()

print("Male rows:", male_diag.shape)
print("Female rows:", female_diag.shape)
male_diag.head(5)

Male rows: (117214, 8)
Female rows: (133788, 8)


,patient_no,node_id,first_date,diagnose_id,descr,icd_code,sex_id,sex
0,9,1325,2015-01-13,1545,GutartigeNeubildung-KurzeKnochenundGelenkknorp...,D161,1,Male
1,9,1974,2015-11-30,2296,"Hyperlipidämie,nichtnäherbezeichnet",E785,1,Male
2,9,3178,2015-11-30,3703,"Essentielle(primäre)Hypertonie,Bluthochdruck,H...",I10,1,Male
3,9,3226,2015-11-30,3761,"ChronischeischämischeHerzkrankheit,nichtnäherb...",I259,1,Male
4,9,3682,2015-11-30,4279,"ChronischeobstruktiveLungenkrankheit,nichtnähe...",J449,1,Male


Instead of incidence matrix the hyperedge table is sotred, where for each patient from the previous table we add hyperedge index.

In [ ]:
def build_hypergraph_tables(df):
    df = df.sort_values(["patient_no", "first_date", "node_id"]).copy()
    df["hyperedge_index"] = df["patient_no"]
# avoiding repetition of gender

    hyperedge_table = df[
        ["hyperedge_index", "patient_no",
         "node_id", "diagnose_id", "descr", "icd_code", "first_date"]
    ].copy()

    patient_sequences = (
        df.groupby(["patient_no", "hyperedge_index"])["node_id"]
        .apply(list)
        .reset_index(name="sequence")
    )

    return hyperedge_table, patient_sequences


In [ ]:
male_hyperedge_table, male_sequences = build_hypergraph_tables(male_diag)
female_hyperedge_table, female_sequences = build_hypergraph_tables(female_diag)

In [ ]:
male_hyperedge_table.head(5)

,hyperedge_index,patient_no,node_id,diagnose_id,descr,icd_code,first_date
0,9,9,1325,1545,GutartigeNeubildung-KurzeKnochenundGelenkknorp...,D161,2015-01-13
1,9,9,1974,2296,"Hyperlipidämie,nichtnäherbezeichnet",E785,2015-11-30
2,9,9,3178,3703,"Essentielle(primäre)Hypertonie,Bluthochdruck,H...",I10,2015-11-30
3,9,9,3226,3761,"ChronischeischämischeHerzkrankheit,nichtnäherb...",I259,2015-11-30
4,9,9,3682,4279,"ChronischeobstruktiveLungenkrankheit,nichtnähe...",J449,2015-11-30


In [ ]:
male_sequences.head(5)

,patient_no,hyperedge_index,sequence
0,9,9,"[1325, 1974, 3178, 3226, 3682, 989, 6834, 475,..."
1,16,16,"[1079, 6927, 348, 71, 10739, 227]"
2,17,17,"[516, 1370, 2669, 3178, 3226, 3425, 9920, 1006..."
3,21,21,"[215, 1139, 932, 8302]"
4,23,23,"[624, 5474, 6734, 250, 3178, 3195, 3219, 3407,..."


Statistics per each hypergraph

In [23]:
def hypergraph_statistics(hyperedge_table):
    n_hyperedges = hyperedge_table["hyperedge_index"].nunique()
    n_nodes = hyperedge_table["node_id"].nunique()
    n_memberships = len(hyperedge_table)
    edge_sizes = (
        hyperedge_table.groupby("hyperedge_index")["node_id"]
        .nunique()
    )

    # how many patients share 1 diagnosis
    node_degrees = (
        hyperedge_table.groupby("node_id")["hyperedge_index"]
        .nunique()
    )

    stats = pd.DataFrame({
        "n_hyperedges": [n_hyperedges],
        "n_nodes": [n_nodes],
        "n_memberships": [n_memberships],

        "avg_hyperedge_size": [edge_sizes.mean()],
        "median_hyperedge_size": [edge_sizes.median()],
        "min_hyperedge_size": [edge_sizes.min()],
        "max_hyperedge_size": [edge_sizes.max()],

        "avg_node_degree": [node_degrees.mean()],
        "median_node_degree": [node_degrees.median()],
        "min_node_degree": [node_degrees.min()],
        "max_node_degree": [node_degrees.max()],
    })

    return stats, edge_sizes, node_degrees
male_stats, male_edge_sizes, male_node_degrees = hypergraph_statistics(male_hyperedge_table)
female_stats, female_edge_sizes, female_node_degrees = hypergraph_statistics(female_hyperedge_table)

male_stats.index = ["Male"]
female_stats.index = ["Female"]

all_stats = pd.concat([male_stats, female_stats])
print(all_stats)

        n_hyperedges  n_nodes  n_memberships  avg_hyperedge_size  \
Male           12867     4926         117214            9.109660   
Female         15402     4994         133788            8.686404   

        median_hyperedge_size  min_hyperedge_size  max_hyperedge_size  \
Male                      6.0                   1                 204   
Female                    6.0                   1                 127   

        avg_node_degree  median_node_degree  min_node_degree  max_node_degree  
Male          23.794965                 4.0                1             1813  
Female        26.789748                 4.0                1             2041  


In [22]:
# top 10 diagnosis by its frequencies

def top_diagnoses(hyperedge_table, top_n=10):
    top_diag = (
        hyperedge_table
        .groupby(["node_id", "diagnose_id", "icd_code"])["hyperedge_index"]
        .nunique()
        .reset_index(name="n_patients")
        .sort_values("n_patients", ascending=False)
        .head(top_n)
    )
    return top_diag

male_top10 = top_diagnoses(male_hyperedge_table, top_n=10)
female_top10 = top_diagnoses(female_hyperedge_table, top_n=10)

print("\nTop male diagnoses")
print(male_top10[["icd_code","n_patients"]])

print("\nTop female diagnoses")
print(female_top10[["icd_code","n_patients"]])


Top male diagnoses
     icd_code  n_patients
817      C449        1813
157      A408        1550
2099      I10        1395
145      A390        1301
433      B407        1276
633      C091        1236
1024     C919        1009
616      C048         956
442      B447         949
428      B392         946

Top female diagnoses
     icd_code  n_patients
816      C449        2041
154      A408        1802
142      A390        1468
427      B407        1465
633      C091        1423
2115      I10        1385
439      B447        1123
615      C048        1077
1022     C919        1071
422      B392        1046
